# Installation and Download


In [1]:
!pip install pdfplumber pydantic openai requests -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 52.5 MB/s eta 0:00:00


In [2]:
import requests
from bs4 import BeautifulSoup

In [3]:
def get_pdf_url(ad_page_url: str) -> str:
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(ad_page_url, headers=headers, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/blob/" in href and ".pdf" in href.lower():
            if href.startswith("http"):
                return href
            return "https://ad.easa.europa.eu" + href

    raise ValueError(f"No PDF link found on {ad_page_url}")

AD_PAGES = {
    "FAA-2025-23-53": "https://ad.easa.europa.eu/ad/US-2025-23-53",
    "EASA-2025-0254": "https://ad.easa.europa.eu/ad/2025-0254R1",
}

PDF_URLS = {}
for ad_id, page_url in AD_PAGES.items():
    PDF_URLS[ad_id] = get_pdf_url(page_url)
    print(ad_id, "->", PDF_URLS[ad_id])

FAA-2025-23-53 -> https://ad.easa.europa.eu/blob/2025-23-53.pdf/AD_US-2025-23-53_1
EASA-2025-0254 -> https://ad.easa.europa.eu/blob/EASA_AD__2025_0254_R1.pdf/AD_2025-0254R1_1


In [4]:
# download

import requests, os

os.makedirs("ads", exist_ok=True)

for ad_id, url in PDF_URLS.items():
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    r.raise_for_status()
    path = f"ads/{ad_id}.pdf"
    with open(path, "wb") as f:
        f.write(r.content)
    print(ad_id, "saved,", len(r.content), "bytes")

FAA-2025-23-53 saved, 210100 bytes
EASA-2025-0254 saved, 227068 bytes


In [5]:
# extract the texts

!pip install pdfplumber -q

import pdfplumber

def extract_text(pdf_path: str) -> str:
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text_parts.append(t)
    return "\n".join(text_parts)

raw_texts = {}
for ad_id in PDF_URLS:
    raw_texts[ad_id] = extract_text(f"ads/{ad_id}.pdf")
    print(f"--- {ad_id} ({len(raw_texts[ad_id])} chars) ---")
    print(raw_texts[ad_id][:300])
    print()

--- FAA-2025-23-53 (15698 chars) ---
[Federal Register, Volume 90 Number 224 (Monday, November 24, 2025)]
[Rules and Regulations]
[Pages 52851-52853]
From the Federal Register Online via the Government Publishing Office [www.gpo.gov]
[FR Doc No: 2025-20804]
DEPARTMENT OF TRANSPORTATION
Federal Aviation Administration
14 CFR Part 39
[Do

--- EASA-2025-0254 (12809 chars) ---
EASA AD No.: 2025-0254R1
Airworthiness Directive
AD No.: 2025-0254R1
Issued: 05 December 2025
Note: This Airworthiness Directive (AD) is issued by EASA, acting in accordance with Regulation
(EU) 2018/1139 on behalf of the European Union, its Member States and of the European third
countries that par



# Pyndantic Scheme

In [6]:
# pyndantic scheme for structured output

from pydantic import BaseModel
from typing import Optional, List

class ModificationExclusion(BaseModel):
    description: str
    type: str

class ApplicabilityRules(BaseModel):
    aircraft_models: List[str]
    msn_constraints: Optional[str] = None
    excluded_if_modifications: List[ModificationExclusion] = []
    required_modifications: List[str] = []
    notes: Optional[str] = None
class ADRules(BaseModel):
    ad_id: str
    subject: str
    applicability_rules: ApplicabilityRules

# LLM for Extraction

In [7]:
from openai import OpenAI
from google.colab import userdata
import json
import re

api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

EXTRACTION_PROMPT = """You are extracting structured applicability rules from an Airworthiness Directive (AD).

Read the AD text below and extract ONLY the applicability section into this exact JSON schema:

{{
  "ad_id": "string",
  "subject": "string - brief description of what the AD is about",
  "applicability_rules": {{
    "aircraft_models": ["list of exact model designations mentioned"],
    "msn_constraints": "string or null - describe any MSN-specific restriction, null if applies to all MSN",
    "excluded_if_modifications": [
      {{"description": "exact mod/SB description that excludes an aircraft", "type": "production_mod|service_bulletin|other"}}
    ],
    "required_modifications": ["list of mods/SBs that must ALREADY be present for the AD to apply, if any"],
    "notes": "string or null - any ambiguity, grouping, or nuance a database engineer should know"
  }}
}}

Rules:
- Only extract from the "Applicability" section (or equivalent, e.g. paragraph (c) in FAA ADs).
- Do not include compliance times, inspection intervals, or corrective actions.
- If applicability is defined in terms of "Groups" (e.g. Group 1, Group 2), resolve what defines each group and fold that into msn_constraints or notes.
- Return ONLY valid JSON, no markdown fences, no preamble.

AD_ID: {ad_id}

AD TEXT:
{ad_text}
"""

def clean_json(raw_text: str) -> str:
    raw_text = (
        raw_text.strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)

    if match:
        return match.group(0)

    return raw_text

extracted = {}

for ad_id, text in raw_texts.items():

    prompt = EXTRACTION_PROMPT.format(
        ad_id=ad_id,
        ad_text=text
    )

    try:

        response = client.chat.completions.create(
            model="google/gemma-4-26b-a4b-it:free",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        raw_json = clean_json(
            response.choices[0].message.content
        )

        parsed = ADRules.model_validate_json(raw_json)

        extracted[ad_id] = parsed

        print(f"{ad_id} extracted successfully")

    except Exception as e:

        print(f"{ad_id} failed")
        print(e)

for ad_id, result in extracted.items():
    print("\n==============================")
    print(ad_id)
    print(result.model_dump_json(indent=2))
    break

FAA-2025-23-53 extracted successfully
EASA-2025-0254 extracted successfully

FAA-2025-23-53
{
  "ad_id": "FAA-2025-23-53",
  "subject": "Engine-pylon structure unsafe condition on Boeing MD-11, MD-10, and DC-10 series airplanes",
  "applicability_rules": {
    "aircraft_models": [
      "MD-11",
      "MD-11F",
      "MD-10-10F",
      "MD-10-30F",
      "DC-10-10",
      "DC-10-10F",
      "DC-10-15",
      "DC-10-30",
      "DC-10-30F (KC-10A and KDC-10)",
      "DC-10-40",
      "DC-10-40F"
    ],
    "msn_constraints": null,
    "excluded_if_modifications": [],
    "required_modifications": [],
    "notes": "The AD applies to all Boeing airplanes in the specified models, certificated in any category. Note that for MD-11 and MD-11F, the prohibition on flight is effective as of December 1, 2025 (the effective date of the superseded Emergency AD 2025-23-51), whereas for the other models, it is effective as of the effective date of this AD."
  }
}


Extracted json to check

In [8]:
print(extracted.keys())

dict_keys(['FAA-2025-23-53', 'EASA-2025-0254'])


In [9]:
for ad_id, rules in extracted.items():
    print("="*40)
    print(ad_id)
    print(rules.model_dump_json(indent=2))

FAA-2025-23-53
{
  "ad_id": "FAA-2025-23-53",
  "subject": "Engine-pylon structure unsafe condition on Boeing MD-11, MD-10, and DC-10 series airplanes",
  "applicability_rules": {
    "aircraft_models": [
      "MD-11",
      "MD-11F",
      "MD-10-10F",
      "MD-10-30F",
      "DC-10-10",
      "DC-10-10F",
      "DC-10-15",
      "DC-10-30",
      "DC-10-30F (KC-10A and KDC-10)",
      "DC-10-40",
      "DC-10-40F"
    ],
    "msn_constraints": null,
    "excluded_if_modifications": [],
    "required_modifications": [],
    "notes": "The AD applies to all Boeing airplanes in the specified models, certificated in any category. Note that for MD-11 and MD-11F, the prohibition on flight is effective as of December 1, 2025 (the effective date of the superseded Emergency AD 2025-23-51), whereas for the other models, it is effective as of the effective date of this AD."
  }
}
EASA-2025-0254
{
  "ad_id": "EASA-2025-0254R1",
  "subject": "ATA 57 – Wing – Main Landing Gear Retraction Actuator

In [10]:
# save to Json

os.makedirs("output", exist_ok=True)
for ad_id, rules in extracted.items():
    with open(f"output/{ad_id}.json", "w") as f:
        f.write(rules.model_dump_json(indent=2))

# Evaluation Logic

In [11]:
import re

class AircraftConfig(BaseModel):
    model: str
    msn: int
    modifications: List[str] = []  # ex: ["mod 24591 (production)", "SB A320-57-1089 Rev 04"]

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower()).strip()

def model_matches(aircraft_model: str, ad_models: List[str]) -> bool:
    am = normalize(aircraft_model)
    return any(normalize(m) == am for m in ad_models)

def has_exclusion(aircraft: AircraftConfig, exclusions: List[ModificationExclusion]) -> bool:
    aircraft_mods_norm = [normalize(m) for m in aircraft.modifications]
    for excl in exclusions:
        excl_norm = normalize(excl.description)
        # substring match check from keyword of mod/SB number
        for am in aircraft_mods_norm:
            if am in excl_norm or excl_norm in am:
                return True
            # match by mod number / SB number
            numbers = re.findall(r"\d{3,}", excl_norm)
            if numbers and any(n in am for n in numbers):
                return True
    return False

def evaluate(aircraft: AircraftConfig, ad: ADRules) -> str:
    rules = ad.applicability_rules
    if not model_matches(aircraft.model, rules.aircraft_models):
        return "Not applicable"
    if has_exclusion(aircraft, rules.excluded_if_modifications):
        return "Not affected"
    # if there's required_modifications but a/c doesnt has -> not affected
    if rules.required_modifications:
        req_norm = [normalize(r) for r in rules.required_modifications]
        aircraft_mods_norm = [normalize(m) for m in aircraft.modifications]
        if not any(r in am or am in r for r in req_norm for am in aircraft_mods_norm):
            return "Not affected"
    return "Affected"

# Validate

In [12]:
# validate using verification examples

verification_cases = [
    (AircraftConfig(model="MD-11F", msn=48400, modifications=[]), "FAA-2025-23-53", "Affected"),
    (AircraftConfig(model="MD-11F", msn=48400, modifications=[]), "EASA-2025-0254", "Not applicable"),
    (AircraftConfig(model="A320-214", msn=4500, modifications=["mod 24591 (production)"]), "FAA-2025-23-53", "Not applicable"),
    (AircraftConfig(model="A320-214", msn=4500, modifications=["mod 24591 (production)"]), "EASA-2025-0254", "Not affected"),
    (AircraftConfig(model="A320-214", msn=4500, modifications=[]), "FAA-2025-23-53", "Not applicable"),
    (AircraftConfig(model="A320-214", msn=4500, modifications=[]), "EASA-2025-0254", "Affected"),
]

for aircraft, ad_id, expected in verification_cases:
    result = evaluate(aircraft, extracted[ad_id])
    status = "ok" if result == expected else "fail"
    print(f"{status} {aircraft.model} MSN{aircraft.msn} vs {ad_id}: got={result} expected={expected}")

ok MD-11F MSN48400 vs FAA-2025-23-53: got=Affected expected=Affected
ok MD-11F MSN48400 vs EASA-2025-0254: got=Not applicable expected=Not applicable
ok A320-214 MSN4500 vs FAA-2025-23-53: got=Not applicable expected=Not applicable
ok A320-214 MSN4500 vs EASA-2025-0254: got=Not affected expected=Not affected
ok A320-214 MSN4500 vs FAA-2025-23-53: got=Not applicable expected=Not applicable
ok A320-214 MSN4500 vs EASA-2025-0254: got=Affected expected=Affected


# Running 10 Test A/C

In [13]:
test_aircraft = [
    AircraftConfig(model="MD-11", msn=48123, modifications=[]),
    AircraftConfig(model="DC-10-30F", msn=47890, modifications=[]),
    AircraftConfig(model="Boeing 737-800", msn=30123, modifications=[]),
    AircraftConfig(model="A320-214", msn=5234, modifications=[]),
    AircraftConfig(model="A320-232", msn=6789, modifications=["mod 24591 (production)"]),
    AircraftConfig(model="A320-214", msn=7456, modifications=["SB A320-57-1089 Rev 04"]),
    AircraftConfig(model="A321-111", msn=8123, modifications=[]),
    AircraftConfig(model="A321-112", msn=364, modifications=["mod 24977 (production)"]),
    AircraftConfig(model="A319-100", msn=9234, modifications=[]),
    AircraftConfig(model="MD-10-10F", msn=46234, modifications=[]),
]

import pandas as pd
rows = []
for a in test_aircraft:
    rows.append({
        "Model": a.model, "MSN": a.msn, "Mods": ", ".join(a.modifications) or "None",
        "FAA 2025-23-53": evaluate(a, extracted["FAA-2025-23-53"]),
        "EASA 2025-0254": evaluate(a, extracted["EASA-2025-0254"]),
    })

df = pd.DataFrame(rows)
df

,Model,MSN,Mods,FAA 2025-23-53,EASA 2025-0254
0,MD-11,48123,None,Affected,Not applicable
1,DC-10-30F,47890,None,Not applicable,Not applicable
2,Boeing 737-800,30123,None,Not applicable,Not applicable
3,A320-214,5234,None,Not applicable,Affected
4,A320-232,6789,mod 24591 (production),Not applicable,Not affected
5,A320-214,7456,SB A320-57-1089 Rev 04,Not applicable,Not affected
6,A321-111,8123,None,Not applicable,Affected
7,A321-112,364,mod 24977 (production),Not applicable,Not affected
8,A319-100,9234,None,Not applicable,Not applicable
9,MD-10-10F,46234,None,Affected,Not applicable


# Export Data

In [14]:
# export

df.to_csv("output/test_results.csv", index=False)
print("Done. Files in output/:", os.listdir("output"))

Done. Files in output/: ['EASA-2025-0254.json', 'FAA-2025-23-53.json', 'test_results.csv']


In [19]:
import shutil, os

os.makedirs("submission", exist_ok=True)
os.makedirs("submission/output", exist_ok=True)

shutil.copy("output/test_results.csv", "submission/output/")
shutil.copy("output/FAA-2025-23-53.json", "submission/output/")
shutil.copy("output/EASA-2025-0254.json", "submission/output/")

print(os.listdir("submission"))
print(os.listdir("submission/output"))

['output']
['EASA-2025-0254.json', 'FAA-2025-23-53.json', 'test_results.csv']


In [20]:
# zip to extract the data

import shutil
shutil.make_archive("submission", "zip", "submission")

from google.colab import files
files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>